# Bag-of-n-Grams

_Feature Engineering (Zheng & Casari)_

**Count word-pairs and word-triples, not just single words — so "not good" can become its own feature.**

Bag-of-words turns a document into a vector of word counts and forgets the order. "dog bites man" and "man bites dog" get the exact same vector. That lost order sometimes carries the whole meaning — especially negation: "good" and "not good" are opposites, but bag-of-words sees the same word "good" in both.

       Bag-of-n-grams recovers a little of that order. Instead of counting only single words, it also counts contiguous runs of $n$ words. Now "not good" is its own feature, separate from "good" and "not". A sentiment model can learn that the "not good" column predicts a bad review.

---

This notebook is a practice scaffold for the **Bag-of-n-Grams** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn (CountVectorizer), pandas

In [ ]:
# pip install scikit-learn pandas
# Dataset: Yelp Dataset Challenge (round 6) reviews, yelp.com/dataset
# Book code: github.com/alicezheng/feature-engineering-book (Chapter 3)
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# --- load the Yelp review text (one JSON object per line) ---
review_file = "yelp_academic_dataset_review.json"
reviews = []
with open(review_file) as f:
    for line in f:
        reviews.append(json.loads(line))
review_df = pd.DataFrame(reviews)
texts = review_df["text"]          # the raw review strings

# --- Bag-of-words: single words only (unigrams) ---
bow = CountVectorizer()            # default ngram_range=(1, 1)
bow.fit(texts)
print("unigram features :", len(bow.get_feature_names_out()))   # ~ 29,000

# --- Bag-of-bigrams: word-pairs only ---
bigram = CountVectorizer(ngram_range=(2, 2))
bigram.fit(texts)
print("bigram features  :", len(bigram.get_feature_names_out())) # ~ 357,000

# --- Bag-of-trigrams: word-triples only ---
trigram = CountVectorizer(ngram_range=(3, 3))
trigram.fit(texts)
print("trigram features :", len(trigram.get_feature_names_out())) # ~ 1,627,000

# --- The useful setting: unigrams AND bigrams together ---
uni_bi = CountVectorizer(ngram_range=(1, 2))
X = uni_bi.fit_transform(texts)    # sparse document x n-gram count matrix
print("uni+bigram feats :", len(uni_bi.get_feature_names_out()))

# negation now has its own feature: "not good" is distinct from "good"
vocab = uni_bi.vocabulary_
print('"good" index     :', vocab.get("good"))
print('"not good" index :', vocab.get("not good"))

# the cost is sparsity: a huge, mostly-zero matrix
print("matrix shape     :", X.shape, " stored nonzeros:", X.nnz)

# tame the explosion: keep only n-grams seen in many reviews
pruned = CountVectorizer(ngram_range=(1, 2), min_df=10, max_features=50000)
pruned.fit(texts)
print("pruned features  :", len(pruned.get_feature_names_out()))

## Visualize the data & results

_On a tiny real corpus of reviews (one with the negation "not good"), how fast does the number of distinct features grow as we widen ngram_range from (1,1) to (1,2) to (1,3)?_

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# six short REAL reviews; one carries a negation ("not good")
corpus = [
    "the food is good",
    "the food is not good",
    "great service and great food",
    "service was not great",
    "i would not come back",
    "good food and good service",
]

# count distinct features for three n-gram ranges
counts = {}
for rng in [(1, 1), (1, 2), (1, 3)]:
    vec = CountVectorizer(ngram_range=rng)
    vec.fit(corpus)
    counts[rng] = len(vec.get_feature_names_out())
    print("ngram_range", rng, "->", counts[rng], "features")
# ngram_range (1, 1) -> 12 features
# ngram_range (1, 2) -> 31 features
# ngram_range (1, 3) -> 45 features
# (sklearn's default token pattern drops 1-char tokens like "i")

# the (1,2) bag now has "not good" as its OWN feature, distinct from "good"
vec12 = CountVectorizer(ngram_range=(1, 2)).fit(corpus)
vocab = vec12.vocabulary_
print('"good" in vocab     :', "good" in vocab)        # True
print('"not good" in vocab :', "not good" in vocab)    # True  <- recovered negation

# the cost: a wider, sparser document x feature matrix
X = vec12.transform(corpus)
print("matrix shape", X.shape, "nonzeros", X.nnz,
      "sparsity %.1f%%" % (100 * (1 - X.nnz / (X.shape[0] * X.shape[1]))))

import matplotlib.pyplot as plt
ranges = ["(1,1)", "(1,2)", "(1,3)"]
vals = [counts[(1, 1)], counts[(1, 2)], counts[(1, 3)]]
plt.bar(ranges, vals, color=["#4ea1ff", "#7ee787", "#ffb454"])
plt.ylabel("number of distinct features")
plt.title("Feature count explodes as ngram_range widens")
for i, v in enumerate(vals):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The review i would not order this again has 6 words. How many bigrams ($n=2$) and trigrams ($n=3$) does it produce? Which bigram carries the negation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $L - n + 1$ with $L = 6$. Bigrams: $6 - 2 + 1 = 5$. Trigrams: $6 - 3 + 1 = 4$. — _Slide a window of width $n$ across the 6 words._
- List the bigrams: "i would", "would not", "not order", "order this", "this again". — _Each adjacent pair is one bigram feature._
- "not order" (and "would not") is the negation-bearing pair. — _Bag-of-words would split "not" from "order"; the bigram keeps the negation as one feature._

**Answer:** 5 bigrams and 4 trigrams. The bigram "not order" (helped by "would not") carries the negation that plain bag-of-words loses.

</details>

**Problem 2.** On Yelp the book reports about 29,000 unigrams, 357,000 bigrams, and 1,627,000 trigrams. Roughly how many times more features do bigrams and trigrams give versus unigrams, and what does that imply about the matrix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bigrams vs unigrams: $357\text{k} / 29\text{k} \approx 12$. Trigrams vs unigrams: $1{,}627\text{k} / 29\text{k} \approx 56$. — _Divide each count by the unigram count to get the multiplier._
- So adding bigrams makes the feature space ~12x wider; trigrams ~56x wider. — _Distinct n-grams grow far faster than the vocabulary._
- Each document still contains only a handful of those n-grams, so almost every column is zero. — _More columns with the same few hits per row means a much sparser matrix._

**Answer:** Bigrams ~12x and trigrams ~56x more features than unigrams. The matrix becomes far wider and much sparser — the cost you pay for the extra local-order signal. Prune with min_df / max_features.

</details>

**Problem 3.** A sentiment model on bag-of-words keeps calling "the food was not good" positive. Why, and what single change to the vectorizer most directly fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bag-of-words ($n=1$) has separate "not" and "good" features; the strong positive weight on "good" wins. — _Unigrams cannot represent that "not" negates "good" — order is discarded._
- Switch to a bag-of-n-grams that includes bigrams: CountVectorizer(ngram_range=(1,2)). — _This adds the "not good" feature, which the model can weight negatively._
- Optionally add min_df to prune the new rare bigrams and control the blow-up. — _Most added bigrams are rare noise; pruning keeps the matrix manageable._

**Answer:** Unigrams split "not" from "good", so the positive "good" weight dominates. Set ngram_range=(1,2) so "not good" becomes its own feature the model can penalize — and use min_df to tame the feature explosion.

</details>